# Colab 24d — feed-matched evaluation tables (supersedes the single `cath_eval.csv.gz` protocol)

**What this fixes.** Every prior eval (colab15 → colab24c) ranked all three feeds (AA, SS, 3Di) against a *single* `cath_eval.csv.gz`. That file was built by an **AA-centred** protocol: it required *both* the AA and SS sequence to be valid, required AA and SS lengths to match, and stratified the selected rows using **AA** similarity bins. colab24c fixed the candidate **pools** and the **oracle** sets, but its query **seeds** still inherited that historical AA-centred selection — so the SS and 3Di feeds were being queried with an AA-shaped sample.

**The fix.** Rebuild the eval *from the raw pair-source files*, **independently per feed**, and write three successor tables. The old `cath_eval.csv.gz` is left **untouched** for legacy-Colab comparison (see `memory/feedback_dataset_rollback.md` — never rewrite a committed dataset; add new files).

| feed | score used | high cut | retrieval queries |
|---|---|---|---|
| AA | recomputed `normLev` on current strings (supplied `aa_score` kept as provenance) | `>= 0.70` | endpoints of high-AA pairs |
| SS | recomputed `normLev` on current strings (supplied `ss_score` kept as provenance) | `>= 0.70` | endpoints of high-SS pairs |
| 3Di | recomputed `normLev` on the two 3Di strings (no supplied score) | `>= 0.70` | endpoints of high-3Di pairs |

**One metric everywhere (P1).** High-pair selection, AUROC labels, the model input, and the full-pool oracle all judge similarity with the *same* recomputed `normLev` on the *current* pool strings. The supplied `aa_score`/`ss_score` differ from a fresh recompute by up to ~0.03 (a different source-string snapshot) and are kept **only** as `*_supplied` provenance columns.

**Scope statement (state it exactly).** *"All high pairs"* means **all high pairs available in the supplied pair-source files after feed-specific filtering** — **not** all possible pairs in CATH. The pair files (`cath_s20_pairs_sample` + `cath_s20_pairs_high`) are a *sample* (~57k unordered pairs), **not** an exhaustive C(14907,2) ≈ 111M enumeration. AA high pairs are genuinely scarce *in this sample* (only 5); the SS / 3Di counts are likewise relative to the sample.

**Expected audit numbers** (reproduced locally 2026-06-23; the notebook prints them live):

| feed | source pairs (deduped) | eligible | high (≥0.70) | query endpoints | pool |
|---|---|---|---|---|---|
| AA | 56,763 | 28,086 | 5 | 10 | ~10,501 |
| SS | 56,763 | 28,059 | 606 | 915 | ~10,497 |
| 3Di | 56,763 | 28,086 | 45 | 57 | ~10,501 |

(High counts use recomputed `normLev` on current strings; the supplied `ss_score` would give 609/921 — three pairs sit just under 0.70 once rescored, which is exactly why we standardise on the oracle's metric.)

The SS feed gains real power: the old AA-centred protocol surfaced ~333 high-SS pairs; the feed-matched protocol surfaces **606** (915 query endpoints). AA stays scarce by nature.

**Comparability (not improvement/regression).** The encoder recipe is byte-identical to colab24c, so the **same model** is re-evaluated under the corrected feed-matched protocol. The evaluation *task* has changed (different source-pair sample, different high-pair query endpoints, different AUROC pair population, SS queries 506→915). Read any difference from colab24c as **sensitivity to the previous AA-centred query selection**, never as the model getting better or worse.

## 1. Setup

In [ ]:
import os
os.chdir('/content')
!rm -rf thesis-edit-distance-nn
!git clone https://github.com/katzemelli/thesis-edit-distance-nn.git
os.chdir('/content/thesis-edit-distance-nn')

In [ ]:
DATA_DIR = '/content/thesis-edit-distance-nn/sampledata/cath'
import os
# Source pair files (the canonical labels) + pools + the legacy eval (kept for comparison only).
for f in ['cath_s20_pairs_sample.csv.gz', 'cath_s20_pairs_high.csv.gz',
          'cath_s20_train70.csv.gz', 'cath_s20_test30.csv.gz',
          'cath_s20_3di.csv.gz', 'cath_eval.csv.gz']:
    p = os.path.join(DATA_DIR, f)
    print(f'{"OK" if os.path.exists(p) else "MISSING":<8} {p}')

In [ ]:
!pip install torch torchvision rapidfuzz scikit-learn scipy --quiet

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score
from rapidfuzz.distance import Levenshtein as RFLevenshtein
from rapidfuzz.process import cdist as rf_cdist

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

## 2. Constants (encoder recipe identical to colab24c / colab19 / colab17b)

In [ ]:
AA_ALPHABET = 'ACDEFGHIKLMNPQRSTVWY'
SS_ALPHABET = 'HLS'
VOCAB_SIZE = len(AA_ALPHABET) + 1
PAD_IDX = len(AA_ALPHABET)
AA_SET = set(AA_ALPHABET)
SS_SET = set(SS_ALPHABET)
CHAR_TO_IDX = {c: i for i, c in enumerate(AA_ALPHABET)}

MIN_LEN = 50
MAX_LEN_SEQ = 200
MAX_LEN = 200

N_TRAIN = 30000
NUM_EPOCHS = 30
BATCH_SIZE = 128
K = 16

BAND_LOW_AA = 0.30
BAND_LOW_SS = 0.56
BAND_HIGH   = 0.70          # high-sim cutoff: positives for AUROC + oracle true-set bar + query selection
BIN_NAMES = ['far', 'mid', 'high']

K_RETRIEVAL = 10
SEED_TRAIN_AA = 42
SEED_TRAIN_SS = 43

print(f'AA encoder BAND_LOW={BAND_LOW_AA} | SS encoder BAND_LOW={BAND_LOW_SS} '
      f'| high-sim cutoff={BAND_HIGH} | retrieval @k={K_RETRIEVAL} | seed={SEED}')

## 3. Helpers — Levenshtein, encode, perturb (RNG threaded for determinism)

In [ ]:
def norm_lev(a, b):
    L = max(len(a), len(b))
    return 1.0 if L == 0 else 1.0 - RFLevenshtein.distance(a, b) / L

def is_standard_aa(seq): return all(c in AA_SET for c in seq)
def is_standard_ss(seq): return all(c in SS_SET for c in seq)

def encode_pad(seq, max_len=MAX_LEN, pad_idx=PAD_IDX):
    if len(seq) > max_len:
        raise ValueError(f'sequence length {len(seq)} exceeds max_len={max_len} '
                         f'(pool is filtered to <= {max_len}; unfiltered input?)')
    idx = [CHAR_TO_IDX[c] for c in seq]
    idx += [pad_idx] * (max_len - len(idx))
    return torch.tensor(idx, dtype=torch.long)

def perturb(seq, k, alphabet, rng, max_len=MAX_LEN):
    s = list(seq); abc = list(alphabet)
    for _ in range(k):
        if len(s) == 0: op = 'ins'
        elif len(s) >= max_len: op = rng.choice(['sub', 'del'])
        else: op = rng.choice(['sub', 'ins', 'del'])
        if op == 'sub':
            i = rng.integers(0, len(s)); choices = [c for c in abc if c != s[i]]; s[i] = rng.choice(choices)
        elif op == 'ins':
            i = rng.integers(0, len(s) + 1); s.insert(i, rng.choice(abc))
        elif op == 'del':
            i = rng.integers(0, len(s)); del s[i]
    return ''.join(s)

def random_seq(alphabet, rng, min_len=MIN_LEN, max_len=MAX_LEN_SEQ):
    length = int(rng.integers(min_len, max_len + 1))
    return ''.join(rng.choice(list(alphabet), size=length))

def bin_idx_for(x, band_low):
    if x < band_low: return 0
    if x < BAND_HIGH: return 1
    return 2

def make_full_range_pairs(alphabet, n_pairs, rng):
    pairs = []; attempts = 0; max_attempts = n_pairs * 4
    while len(pairs) < n_pairs and attempts < max_attempts:
        attempts += 1
        seed = random_seq(alphabet, rng)
        if len(seed) < MIN_LEN: continue
        L = len(seed); t = float(rng.uniform(0.0, 1.0)); k = max(0, int(round((1.0 - t) * L)))
        other = perturb(seed, k, alphabet, rng)
        if 1 <= len(other) <= MAX_LEN:
            pairs.append((seed, other, norm_lev(seed, other)))
    return pairs

## 4. Architecture + training (3-bin CE head, identical to colab24c)

In [ ]:
class SiameseEncoder(nn.Module):
    def __init__(self, K, vocab_size=VOCAB_SIZE, embed_dim=32,
                 hidden1=32, hidden2=64, out_dim=128, pad_idx=PAD_IDX):
        super().__init__()
        self.K = K; self.pad_idx = pad_idx
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)
        self.conv1 = nn.Conv1d(embed_dim, hidden1, kernel_size=3, padding=1)
        self.conv2 = nn.Conv1d(hidden1, hidden2, kernel_size=3, padding=1)
        self.pool  = nn.AdaptiveAvgPool1d(K)
        self.fc    = nn.Linear(hidden2 * K, out_dim)

    def forward(self, x):
        mask = (x != self.pad_idx).float()
        e = self.embedding(x).permute(0, 2, 1)
        h = F.relu(self.conv1(e)); h = F.relu(self.conv2(h))
        h = h * mask.unsqueeze(1)
        h = self.pool(h).flatten(1)
        return F.normalize(self.fc(h), p=2, dim=1)


class SiameseClassifier(nn.Module):
    def __init__(self, K, embed_out=128, hidden_mlp=64, n_bins=3):
        super().__init__()
        self.encoder = SiameseEncoder(K)
        self.head = nn.Sequential(
            nn.Linear(embed_out, hidden_mlp), nn.LeakyReLU(0.01),
            nn.Linear(hidden_mlp, n_bins))
    def forward(self, a, b):
        return self.head(torch.abs(self.encoder(a) - self.encoder(b)))
    def encode(self, x):
        return self.encoder(x)


class PairDatasetCE(Dataset):
    def __init__(self, pairs, band_low):
        self.a = [encode_pad(a) for a, _, _ in pairs]
        self.b = [encode_pad(b) for _, b, _ in pairs]
        self.bins = torch.tensor([bin_idx_for(l, band_low) for _, _, l in pairs], dtype=torch.long)
    def __len__(self): return len(self.bins)
    def __getitem__(self, i): return self.a[i], self.b[i], self.bins[i]


def train_encoder(alphabet, band_low, label, train_seed, n_epochs=NUM_EPOCHS):
    torch.manual_seed(train_seed)
    rng = np.random.default_rng(train_seed)
    print(f'\n===== Training {label} encoder (alphabet={alphabet}, band_low={band_low}, seed={train_seed}) =====')
    pairs = make_full_range_pairs(alphabet, N_TRAIN, rng)
    labels = np.array([p[2] for p in pairs])
    counts = {b: int(c) for b, c in zip(BIN_NAMES,
              np.bincount([bin_idx_for(l, band_low) for l in labels], minlength=3))}
    print(f'  {len(pairs)} pairs, normLev in [{labels.min():.3f}, {labels.max():.3f}], bins={counts}')
    dl = DataLoader(PairDatasetCE(pairs, band_low), batch_size=BATCH_SIZE, shuffle=True)
    model = SiameseClassifier(K).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    crit = nn.CrossEntropyLoss()
    model.train()
    for epoch in range(1, n_epochs + 1):
        tot = 0.0; nb = 0
        for a, b, y in dl:
            a, b, y = a.to(device), b.to(device), y.to(device)
            loss = crit(model(a, b), y)
            opt.zero_grad(); loss.backward(); opt.step()
            tot += loss.item(); nb += 1
        if epoch % 5 == 0 or epoch == 1:
            print(f'  [{label}] epoch {epoch:3d}/{n_epochs} CE={tot/nb:.4f}')
    model.eval()
    return model

## 5. Per-representation pools (genuinely per-feed, identical to colab24c)

Each representation is filtered **only** by its own validity (standard alphabet + length in `[MIN_LEN, MAX_LEN]`); a domain's eligibility in one representation never depends on another. `RESCUED` keeps the sub-`MIN_LEN` high-AA pair wherever its sequence is otherwise valid.

In [ ]:
train_df = pd.read_csv(os.path.join(DATA_DIR, 'cath_s20_train70.csv.gz'))
test_df  = pd.read_csv(os.path.join(DATA_DIR, 'cath_s20_test30.csv.gz'))
raw = pd.concat([train_df, test_df], ignore_index=True).drop_duplicates('domain_id')
seqs3 = pd.read_csv(os.path.join(DATA_DIR, 'cath_s20_3di.csv.gz'))

RESCUED = {'4z0mC02', '3qkaE02'}

def _valid(seq, is_std, domain):
    return (isinstance(seq, str) and is_std(seq)
            and ((MIN_LEN <= len(seq) <= MAX_LEN) or domain in RESCUED))

id_to_aa  = {d: s for d, s in zip(raw['domain_id'],   raw['aa_seq'])            if _valid(s, is_standard_aa, d)}
id_to_ss  = {d: s for d, s in zip(raw['domain_id'],   raw['ss_seq'])            if _valid(s, is_standard_ss, d)}
id_to_3di = {d: s for d, s in zip(seqs3['domain_id'], seqs3['3di'].astype(str)) if _valid(s, is_standard_aa, d)}

POOL_IDS  = {'AA': list(id_to_aa), 'SS': list(id_to_ss), '3Di': list(id_to_3di)}
LOOKUP    = {'AA': id_to_aa,       'SS': id_to_ss,       '3Di': id_to_3di}
SCORE_COL = {'AA': 'aa_score',     'SS': 'ss_score',     '3Di': '3di_score'}
POOL_SET  = {f: set(POOL_IDS[f]) for f in POOL_IDS}

for f in ['AA', 'SS', '3Di']:
    print(f'  {f:<4} pool = {len(POOL_IDS[f]):>6}')

## 6. Feed-matched eval tables — *the fix*

Load the two raw pair-source files, combine, **deduplicate as unordered pairs** (so `(A,B)` and `(B,A)` cannot both occur), then build three eval tables **independently** — each filtered to *its own* pool and scored in *its own* alphabet. No AA bin is allowed to select an SS or 3Di pair.

> **Column note.** The pair-source files are **headerless** — the first data row was historically mis-read as a header. Column order verified against `cath_eval.csv.gz` (correlation 1.000, zero max-diff): **col2 = `ss_score`, col3 = `aa_score`**. The remaining columns (`src_*`) are the source files' own *local-alignment* metrics and are **not** used (cf. `memory/data_provenance_census.md`: the source `_LEV_filtered` alignment_score is a local metric ≠ our global normLev). We score **every** feed by recomputing `normLev` on the current pool strings — the same metric the oracle uses — and keep the supplied `aa_score`/`ss_score` only as `*_supplied` provenance columns.

In [ ]:
# --- load + combine + dedup unordered ---
PAIR_COLS = ['domain_a', 'domain_b', 'ss_score', 'aa_score', 'src4', 'src5', 'src_flag']
samp = pd.read_csv(os.path.join(DATA_DIR, 'cath_s20_pairs_sample.csv.gz'), header=None, names=PAIR_COLS)
high = pd.read_csv(os.path.join(DATA_DIR, 'cath_s20_pairs_high.csv.gz'), header=None, names=PAIR_COLS)

src_pairs = pd.concat([samp, high], ignore_index=True)
N_SOURCE_RAW = len(src_pairs)
src_pairs = src_pairs[src_pairs['domain_a'] != src_pairs['domain_b']]   # defensive: drop self-pairs
src_pairs['pair_key'] = [frozenset((a, b)) for a, b in zip(src_pairs['domain_a'], src_pairs['domain_b'])]
src_pairs = src_pairs.drop_duplicates('pair_key').reset_index(drop=True)
N_SOURCE_DEDUP = len(src_pairs)
print(f'source pairs: raw concat = {N_SOURCE_RAW:,}  ->  unordered-deduped = {N_SOURCE_DEDUP:,}')

# sanity check the column mapping against the legacy eval (corr should be 1.000, max-diff 0.0)
_ev = pd.read_csv(os.path.join(DATA_DIR, 'cath_eval.csv.gz'))
_ev['pair_key'] = [frozenset((a, b)) for a, b in zip(_ev['domain_a'], _ev['domain_b'])]
_m = _ev.merge(src_pairs, on='pair_key', suffixes=('_ev', ''))
print(f'column-mapping check vs cath_eval (n={len(_m)}):')
print(f"  aa: corr={_m['aa_score_ev'].corr(_m['aa_score']):.4f}  max|diff|={(_m['aa_score_ev']-_m['aa_score']).abs().max():.4f}")
print(f"  ss: corr={_m['ss_score_ev'].corr(_m['ss_score']):.4f}  max|diff|={(_m['ss_score_ev']-_m['ss_score']).abs().max():.4f}")
print('  (expect corr ~1.0000 and max|diff| ~0.0000 -> col2=ss_score, col3=aa_score confirmed)')

In [ ]:
# --- build the three feed eval tables, independently ---
# CANONICAL METRIC (P1): exact normLev recomputed on the CURRENT pool strings, for ALL feeds (AA, SS, 3Di).
# This is the same metric the full-pool oracle uses, so high-pair selection, AUROC labels, the model input,
# and the retrieval oracle all judge similarity identically on the same strings. The supplied aa_score /
# ss_score (which differ from a fresh recompute by up to ~0.03 -- a different source-string snapshot) are
# kept ONLY as '*_supplied' provenance columns, never used for selection.
SUPPLIED_COL = {'AA': 'aa_score', 'SS': 'ss_score'}   # 3Di has no supplied score in the pair files

def build_feed_eval(feed):
    """Eligible = both endpoints in this feed's pool. Score = recomputed normLev on current strings.
    'is_high' = score >= BAND_HIGH. Independent per feed; no cross-feed bin selection."""
    inpool = POOL_SET[feed]; lk = LOOKUP[feed]
    sub = src_pairs[src_pairs['domain_a'].isin(inpool) & src_pairs['domain_b'].isin(inpool)]
    a = sub['domain_a'].values; b = sub['domain_b'].values
    recomputed = np.array([norm_lev(lk[x], lk[y]) for x, y in zip(a, b)])
    out = pd.DataFrame({'domain_a': a, 'domain_b': b, SCORE_COL[feed]: recomputed})
    if feed in SUPPLIED_COL:                                       # provenance only -- not used for selection
        out[SCORE_COL[feed] + '_supplied'] = sub[SUPPLIED_COL[feed]].astype(float).values
    out['is_high'] = (out[SCORE_COL[feed]] >= BAND_HIGH).astype(int)
    return out.reset_index(drop=True)

EVAL = {feed: build_feed_eval(feed) for feed in ['AA', 'SS', '3Di']}

# --- export successor tables (legacy cath_eval.csv.gz is left untouched) ---
OUT_NAME = {'AA': 'cath_eval_aa_perrep.csv.gz',
            'SS': 'cath_eval_ss_perrep.csv.gz',
            '3Di': 'cath_eval_3di_perrep.csv.gz'}
for feed in ['AA', 'SS', '3Di']:
    path = os.path.join(DATA_DIR, OUT_NAME[feed])
    EVAL[feed].to_csv(path, index=False)
    print(f'wrote {OUT_NAME[feed]:<28} rows={len(EVAL[feed]):>6}  high={int(EVAL[feed]["is_high"].sum()):>5}  -> {path}')
print('\nNOTE: also download these from the Colab file browser if you want them committed to the repo'
      ' (do NOT overwrite cath_eval.csv.gz — these are new files; see memory/feedback_dataset_rollback.md).')

## 7. Audit table + scope statement

Per-feed: source-pair count, eligible-pair count, high-pair count, unique query count, pool size, seed. Plus an old-vs-new comparison so the SS power gain is explicit.

In [ ]:
def feed_high(feed):
    e = EVAL[feed]
    return e[e['is_high'] == 1]

def feed_queries(feed):
    h = feed_high(feed)
    return sorted(set(h['domain_a']) | set(h['domain_b']))

print('=' * 96)
print('AUDIT — feed-matched eval construction (all counts are AFTER feed-specific filtering of the')
print('         supplied pair-source SAMPLE; NOT all C(14907,2) possible CATH pairs)')
print('=' * 96)
print(f'{"feed":<6}{"source_pairs":>14}{"eligible":>11}{"high>=0.70":>12}{"queries":>9}{"pool":>9}{"seed":>6}')
print('-' * 96)
QUERIES = {}
for feed in ['AA', 'SS', '3Di']:
    q = feed_queries(feed); QUERIES[feed] = q
    print(f'{feed:<6}{N_SOURCE_DEDUP:>14,}{len(EVAL[feed]):>11,}{int(feed_high(feed).shape[0]):>12,}'
          f'{len(q):>9,}{len(POOL_IDS[feed]):>9,}{SEED:>6}')
print('-' * 96)
print('Scope: "all high pairs" = all high pairs in the supplied pair-source files after feed-specific')
print('       filtering. The pair files are a SAMPLE (~57k unordered pairs), not an exhaustive enumeration.')

# --- old vs new: what the AA-centred cath_eval surfaced per feed ---
print('\n--- legacy cath_eval.csv.gz vs feed-matched (high-pair / unique-query counts) ---')
_ev3 = pd.read_csv(os.path.join(DATA_DIR, 'cath_eval.csv.gz'))
_ev3['3di_score'] = [norm_lev(id_to_3di[a], id_to_3di[b]) if a in id_to_3di and b in id_to_3di else np.nan
                     for a, b in zip(_ev3['domain_a'], _ev3['domain_b'])]
print(f'{"feed":<6}{"legacy high":>13}{"legacy queries":>16}{"new high":>10}{"new queries":>13}')
for feed in ['AA', 'SS', '3Di']:
    sc = SCORE_COL[feed]
    le = _ev3.dropna(subset=[sc])
    le = le[le['domain_a'].isin(POOL_SET[feed]) & le['domain_b'].isin(POOL_SET[feed])]
    lh = le[le[sc] >= BAND_HIGH]
    lq = sorted(set(lh['domain_a']) | set(lh['domain_b']))
    print(f'{feed:<6}{lh.shape[0]:>13,}{len(lq):>16,}{feed_high(feed).shape[0]:>10,}{len(QUERIES[feed]):>13,}')

## 8. Train the two encoders (frozen afterwards)

In [ ]:
model_aa = train_encoder(AA_ALPHABET, BAND_LOW_AA, 'AA', SEED_TRAIN_AA)
model_ss = train_encoder(SS_ALPHABET, BAND_LOW_SS, 'SS', SEED_TRAIN_SS)

## 9. Metric machinery — predicted L2-sim, AUROC over each feed's own eligible set

In [ ]:
@torch.no_grad()
def encode_pool(model, seq_lookup, ids, batch_size=256):
    model.eval(); valid = [d for d in ids if d in seq_lookup]; out = []
    for i in range(0, len(valid), batch_size):
        x = torch.stack([encode_pad(seq_lookup[d]) for d in valid[i:i+batch_size]]).to(device)
        out.append(model.encoder(x))
    return (torch.cat(out, 0) if out else torch.empty(0)), valid

@torch.no_grad()
def pred_sim_strings(model, A, B, batch=512):
    model.eval(); sims = []
    for i in range(0, len(A), batch):
        xa = torch.stack([encode_pad(s) for s in A[i:i+batch]]).to(device)
        xb = torch.stack([encode_pad(s) for s in B[i:i+batch]]).to(device)
        d = torch.linalg.vector_norm(model.encoder(xa) - model.encoder(xb), dim=1)
        sims.append((1.0 - d / 2.0).cpu().numpy())
    return np.concatenate(sims) if sims else np.array([])

def auroc_cell(model, feed):
    """is-high AUROC over THIS feed's full eligible set, using THIS feed's own score bins."""
    sc, lk = SCORE_COL[feed], LOOKUP[feed]
    sub = EVAL[feed]
    y = sub['is_high'].values
    if y.sum() == 0 or y.sum() == len(y):
        return np.nan, int(y.sum()), len(y)
    pred = pred_sim_strings(model, [lk[d] for d in sub['domain_a']], [lk[d] for d in sub['domain_b']])
    return float(roc_auc_score(y, pred)), int(y.sum()), len(y)

## 10. Oracle — full-pool exact-Levenshtein neighbour sets (unchanged from colab24c)

The full-pool exact-Levenshtein oracle *stays as it is*: it independently recomputes the true `>= 0.70` neighbour set per query over the whole pool. (Queries now come from the feed-matched high pairs of §6.)

In [ ]:
def build_oracle_cache(feed):
    """Exact-Levenshtein true neighbour sets (>= BAND_HIGH) per query, over the full pool."""
    lk = LOOKUP[feed]
    pool_ids = POOL_IDS[feed]
    idx = {d: i for i, d in enumerate(pool_ids)}
    pool_seqs = [lk[d] for d in pool_ids]
    lens = np.array([len(s) for s in pool_seqs])
    q_ids = QUERIES[feed]
    if not q_ids:
        return {'pool_ids': pool_ids, 'idx': idx, 'q_ids': [], 'true_sets': {}}
    D = rf_cdist([lk[q] for q in q_ids], pool_seqs,
                 scorer=RFLevenshtein.distance, workers=-1).astype(float)
    true_sets = {}
    for r, q in enumerate(q_ids):
        qi = idx[q]
        denom = np.maximum(lens, lens[qi]); denom[denom == 0] = 1
        osim = 1.0 - D[r] / denom; osim[qi] = -np.inf
        true_sets[q] = set(np.where(osim >= BAND_HIGH)[0].tolist())
    nt = [len(true_sets[q]) for q in q_ids]
    print(f'  oracle[{feed:<3}]: {len(q_ids):>3} queries, median |T|={int(np.median(nt))}, max={max(nt)}')
    return {'pool_ids': pool_ids, 'idx': idx, 'q_ids': q_ids, 'true_sets': true_sets}

print('Building exact-Levenshtein oracle neighbour sets (feed-matched queries)...')
ORACLE = {feed: build_oracle_cache(feed) for feed in ['AA', 'SS', '3Di']}

# Invariant (P1): selection and the oracle now use the SAME normLev metric, so every feed-matched query
# (an endpoint of a >=0.70 pair whose partner is in the pool) MUST have at least one oracle true neighbour.
for feed in ['AA', 'SS', '3Di']:
    ts = ORACLE[feed]['true_sets']
    empty = [q for q in ORACLE[feed]['q_ids'] if len(ts[q]) == 0]
    assert not empty, f'{feed}: {len(empty)} query/queries have an EMPTY oracle true set (metric mismatch): {empty[:5]}'
    print(f'  [{feed:<3}] all {len(ORACLE[feed]["q_ids"]):>3} queries have >= 1 oracle neighbour  OK')

## 11. Retrieval metrics — per-query + bootstrap 95% CI (seed-fixed)

In [ ]:
K_LIST = (10, 50)
K_MAP  = 10
N_BOOT = 2000

def _ap_at_k(order, true_set, k):
    nt = len(true_set)
    if nt == 0: return np.nan
    hits = 0; ap = 0.0
    for i, oi in enumerate(order[:k], start=1):
        if oi in true_set:
            hits += 1; ap += hits / i
    return ap / min(nt, k)

def per_query_metrics(sim_fn, cache, k_list=K_LIST, k_map=K_MAP):
    idx = cache['idx']
    keys = ['ap', 'rprec', 'nt'] + [f'{m}@{k}' for k in k_list for m in ('rec', 'prec', 'hit')]
    per = {kk: [] for kk in keys}
    for q in cache['q_ids']:
        qi = idx[q]; ts = cache['true_sets'][q]; nt = len(ts)
        if nt == 0: continue
        sim = np.asarray(sim_fn(qi), dtype=float).copy(); sim[qi] = -np.inf
        order = np.argsort(-sim)
        per['nt'].append(nt)
        per['ap'].append(_ap_at_k(order, ts, k_map))
        per['rprec'].append(len(set(order[:nt].tolist()) & ts) / nt)
        for k in k_list:
            ret = len(set(order[:k].tolist()) & ts)
            per[f'rec@{k}'].append(ret / nt)
            per[f'prec@{k}'].append(ret / k)
            per[f'hit@{k}'].append(1.0 if ret >= 1 else 0.0)
    return {kk: np.asarray(v, float) for kk, v in per.items()}

def summarize_ci(per, n_boot=N_BOOT, seed=SEED):
    rng = np.random.default_rng(seed)
    n = len(per['nt'])
    out = {'n_q': n, 'median_n_true': float(np.median(per['nt'])) if n else np.nan}
    fields = {f'MAP@{K_MAP}': 'ap', 'Rprec': 'rprec', 'precision@10': 'prec@10',
              'recall@10': 'rec@10', 'recall@50': 'rec@50', 'hitrate@10': 'hit@10'}
    boot_idx = rng.integers(0, n, size=(n_boot, n)) if n >= 2 else None
    for name, kk in fields.items():
        arr = per[kk]; out[name] = float(arr.mean()) if n else np.nan
        if boot_idx is not None:
            b = arr[boot_idx].mean(axis=1)
            out[name + '_lo'] = float(np.percentile(b, 2.5))
            out[name + '_hi'] = float(np.percentile(b, 97.5))
        else:
            out[name + '_lo'] = out[name + '_hi'] = np.nan
    return out

def encoder_pool_np(model, feed):
    pe, _ = encode_pool(model, LOOKUP[feed], ORACLE[feed]['pool_ids'])
    return pe.cpu().numpy()

def length_pool(feed):
    return np.array([len(LOOKUP[feed][d]) for d in ORACLE[feed]['pool_ids']], dtype=float)

## 12. Compute — AUROC + encoder retrieval (with CI) + length-only baseline

In [ ]:
ENCODERS = [('AA-enc', model_aa), ('SS-enc', model_ss)]
FEEDS = ['AA', 'SS', '3Di']
results = []

for enc_label, model in ENCODERS:
    for feed in FEEDS:
        auroc, n_pos, n_tot = auroc_cell(model, feed)
        emb = encoder_pool_np(model, feed)
        per = per_query_metrics(lambda qi, e=emb: 1.0 - np.linalg.norm(e - e[qi], axis=1) / 2.0, ORACLE[feed])
        s = summarize_ci(per)
        row = {'encoder': enc_label, 'feed': feed,
               'role': 'in-domain' if (enc_label, feed) in [('AA-enc', 'AA'), ('SS-enc', 'SS')] else 'cross-rep',
               'auroc': auroc, 'auroc_n_pos': n_pos, 'auroc_n_total': n_tot}
        row.update(s); results.append(row)
        print(f"{enc_label} x {feed:<4} | AUROC={auroc:.3f} (n_pos={n_pos}) | "
              f"MAP@10={s['MAP@10']:.3f} [{s['MAP@10_lo']:.3f},{s['MAP@10_hi']:.3f}] | "
              f"hit@10={s['hitrate@10']:.3f} | med|T|={s['median_n_true']:.0f} (n_q={s['n_q']})")

print('\n--- length-only baseline ---')
for feed in FEEDS:
    plen = length_pool(feed)
    per = per_query_metrics(lambda qi, p=plen: 1.0 - np.abs(p - p[qi]) / np.maximum(p, p[qi]), ORACLE[feed])
    s = summarize_ci(per)
    row = {'encoder': 'length-only', 'feed': feed, 'role': 'baseline',
           'auroc': np.nan, 'auroc_n_pos': 0, 'auroc_n_total': 0}
    row.update(s); results.append(row)
    print(f"length   x {feed:<4} | MAP@10={s['MAP@10']:.3f} [{s['MAP@10_lo']:.3f},{s['MAP@10_hi']:.3f}] | "
          f"hit@10={s['hitrate@10']:.3f} | med|T|={s['median_n_true']:.0f}")

res_df = pd.DataFrame(results)

## 13. Summary table + CSV (feed-matched results of record)

In [ ]:
print('=' * 128)
print('COLAB24d SUMMARY — feed-matched retrieval (bootstrap 95% CI) + length-only baseline; exact-oracle relevance')
print('=' * 128)
print(f'{"enc":<12}{"feed":<6}{"role":<11}{"AUROC":>7}{"n_pos":>6}{"MAP@10 [95% CI]":>24}'
      f'{"hit@10":>9}{"rec@50":>8}{"med|T|":>8}{"n_q":>6}')
print('-' * 128)
for r in results:
    a = f'{r["auroc"]:.3f}' if not np.isnan(r["auroc"]) else '-'
    mp = f'{r["MAP@10"]:.3f} [{r["MAP@10_lo"]:.3f},{r["MAP@10_hi"]:.3f}]'
    print(f'{r["encoder"]:<12}{r["feed"]:<6}{r["role"]:<11}{a:>7}{r["auroc_n_pos"]:>6}{mp:>24}'
          f'{r["hitrate@10"]:>9.3f}{r["recall@50"]:>8.3f}{r["median_n_true"]:>8.0f}{r["n_q"]:>6}')
res_df.to_csv('colab24d_summary.csv', index=False)
print('\nSaved: colab24d_summary.csv  +  three eval tables in sampledata/cath/ '
      '(cath_eval_{aa,ss,3di}_perrep.csv.gz)')

## How to read this notebook

- **§6 is the deliverable**: three feed-matched eval tables, each built independently from the raw pair-source files, scored in its own alphabet, with high pairs and queries defined *per feed*. The legacy `cath_eval.csv.gz` is untouched.
- **§7 audit** is the receipt: it reproduces the locally-validated counts (AA 5 high / 10 queries; SS **606 / 915**; 3Di 45 / 57) and shows the SS power gain over the old AA-centred protocol.
- **§12–13** re-evaluate the **same model** under the corrected feed-matched protocol. The evaluation *task* changed (different source-pair sample, query endpoints, AUROC population; SS queries 506→915), so read any difference from `colab24_summary.csv` as **sensitivity to the previous AA-centred query selection**, not as model improvement or regression.
- **Scope discipline**: every count is relative to the supplied pair-file *sample*, never to all of CATH. AA high pairs are scarce *in this sample* — that scarcity is real, not a protocol artifact.
- **One metric everywhere (P1)**: selection, AUROC labels, model input, and the oracle all use the same recomputed `normLev` on the current strings; §10 asserts every query has ≥1 oracle neighbour. The supplied `aa_score`/`ss_score` are retained only as `*_supplied` provenance columns — they differ from the recompute by up to ~0.03, which is why three SS pairs near the 0.70 cut drop out (supplied 609/921 → recomputed 606/915).